In [34]:
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import os
import random
from geopy.distance import geodesic
from plotly import express as px
import json 
import plotly.graph_objects as go
import geopandas as gpd


## Plotting helper functions

In [2]:
# Function to extract coordinates from GeoJSON and draw boundaries
def add_geojson_boundaries(fig, geojson):
    for feature in geojson['features']:
        geometry = feature['geometry']
        coords = []
        
        if geometry['type'] == 'Polygon':
            coords = geometry['coordinates'][0]  # Outer boundary
        elif geometry['type'] == 'MultiPolygon':
            for polygon in geometry['coordinates']:
                coords.extend(polygon[0])
                coords.append([None, None])  # Separator for discontinuous lines
        
        
        if coords:
            lons, lats = zip(*coords)
            fig.add_trace(go.Scattergeo(
                lon=lons,
                lat=lats,
                mode='lines',
                line=dict(width=1, color='lightblue'),
                showlegend=False,
                hoverinfo='skip'
            ))


# National population (by state)

In [ ]:
# downlaoded from : https://www.inegi.org.mx/sistemas/Olap/Proyectos/bd/censos/cpv2020/pt.asp
mexico_states = {
    # "Aguascalientes": "01",
    "Baja California": "02", "Baja California Sur": "03","Campeche": "04", "Coahuila de Zaragoza": "05","Colima": "06", "Chiapas": "07","Chihuahua": "08","Ciudad de México": "09","Durango": "10","Guanajuato": "11","Guerrero": "12","Hidalgo": "13","Jalisco": "14","México": "15", "Michoacán de Ocampo": "16", "Morelos": "17", "Nayarit": "18", "Nuevo León": "19", "Oaxaca": "20","Puebla": "21", "Querétaro": "22", "Quintana Roo": "23", "San Luis Potosí": "24", "Sinaloa": "25", "Sonora": "26", "Tabasco": "27", "Tamaulipas": "28", "Tlaxcala": "29", "Veracruz de Ignacio de la Llave": "30", "Yucatán": "31", "Zacatecas": "32"
}


one_state = False
first_rows = []
if one_state:
    state = 'San Luis Potosí'
    filename = f'../Datos Nacionales/POBLACION/ITER_{mexico_states[state]}XLSX20.xlsx'
    pop_df = pd.read_excel(filename)

else:
    # TODO: annoyingly inefficient. should probably just save as its own excel file and then load that in.
    for state, code in mexico_states.items():
        df = pd.read_excel(f'../Datos Nacionales/POBLACION/ITER_{mexico_states[state]}XLSX20.xlsx')
        fr = df.iloc[0]
        first_rows.append(fr)
    pop_df = pd.DataFrame(first_rows)
    pop_df = pop_df.reset_index(drop=True)
    
    gdf = gpd.read_file('../Datos SLP/shapefiles/mexico_by_estado.json')
    gdf = gdf.to_crs(6372)
    gdf['centroid'] = gdf['geometry'].centroid
    pop_df['X'] = gdf['centroid'].x
    pop_df['Y'] = gdf['centroid'].y
    
# print(state_pop_df.head())
print(pop_df.head())

# Combined contaminant data

In [ ]:
contaminants_xl = pd.ExcelFile('../Datos Nacionales/BaseNacional-Ladr-Agua-Mina.xlsx')
cont_ladr = pd.read_excel(contaminants_xl,'Ladrilleras')
cont_ladr = cont_ladr.rename(columns={'x': 'X', 'y':'Y', 'municipio': 'MUNICIPIO'})
cont_ladr['tipo'] = 'ladrillera'
cont_minas = pd.read_excel(contaminants_xl,'Minas')
cont_minas['tipo'] = 'mina'
cont_minas = cont_minas.rename(columns={'Municipio': 'MUNICIPIO'})
cont_agua = pd.read_excel(contaminants_xl,'Agua')
cont_agua['tipo'] = 'agua'
cols_to_replace = ['Acrilonitrilo', 'Benceno', 'Arsenico_Total', 'Cadmio_Total', 'Arsenico_Soluble', 'Cadmio_Soluble','metales', 'elementos']


contaminants_all = pd.concat([cont_ladr, cont_agua, cont_minas], ignore_index=True)
contaminants_all[cols_to_replace] = contaminants_all[cols_to_replace].fillna(0)



In [27]:
# downlaoded from https://github.com/PhantomInsights/mexico-geojson/blob/main/2023/%60mexico.zip
with open('../Datos SLP/shapefiles/mexico_by_estado.json', 'r') as f:
    mex_geojson = json.load(f)

In [ ]:
# Create figure
fig = go.Figure()

# Add GeoJSON boundaries
add_geojson_boundaries(fig, mex_geojson)

color_col = 'tipo'
# Add scatter points
for eff in contaminants_all[color_col].unique():
    mun_data = contaminants_all[contaminants_all[color_col] == eff]
    # print(mun_data)
    fig.add_trace(go.Scattergeo(
        lon=mun_data['X'],
        lat=mun_data['Y'],
        text=[mun_data['MUNICIPIO']],
        name=eff,
        mode='markers',
        marker=dict(size=8), 
        opacity=0.5
    ))

fig.update_geos(
    fitbounds="locations",
    visible=True
)

fig.update_layout(
    height=600,
    showlegend=True,
    title_text='Admissions by state'
)


fig.update_layout(title_text='Mapa de ladrilleras, minas, y agua')
fig.show()
# fig.write_html('point_sources.html')

# Computing effects

In [ ]:
import pdb
def pollutant_effect(pollutant, location, pol_name='ALC_mg/L', lon_name = 'X', lat_name = 'Y', pop_name = 'POBTOT'):
    # pdb.set_trace()
    man_loc = (location[lat_name], location[lon_name])
    
    pop = location[pop_name]
    pol_loc = (pollutant[lat_name], pollutant[lon_name])
    distance = geodesic(man_loc, pol_loc).kilometers
    pol_amt = pollutant[pol_name]
    if isinstance(pol_amt, str):
        # if a measurement is "<0.00065", take the measurement as 0.00065, for example
        pol_amt = float(pol_amt[1:])
    
    
    # TODO: revisit if we want to make this more complex
    return pop * pol_amt / distance



pollutant_list = ['Acrilonitrilo', 'Benceno']#, 'Arsenico_Total', 'Cadmio_Total', 'Arsenico_Soluble', 'Cadmio_Soluble']

effects_df = pd.DataFrame()

effects_df['NOM_ENT'] = pop_df['NOM_ENT']

for pol in pollutant_list:
    effects_df[pol] = 0.0


for pol in pollutant_list:
    idx = 0
    for i, location in pop_df.iterrows():
        effect_counter = 0


        for j, measurement in contaminants_all.iterrows():
            # pdb.set_trace()

            effect = pollutant_effect(measurement, location, pol_name = pol)
            effect_counter += effect
        
        effects_df.loc[idx, pol] = effect_counter
        print(f'Effect of {pol} on {location["NOM_ENT"]} is {effect_counter}')
        idx += 1
        
# print(effects_df.head())

# Heatmap of effects of pollutants

In [ ]:

fig = go.Figure()
pollutant = 'Benceno'# 'Acrilonitrilo'
fig = px.choropleth(
    effects_df, 
    geojson=mex_geojson, 
    color=pollutant, 
    locations='NOM_ENT',  # column in state_pop_df that matches GeoJSON features
    featureidkey='properties.name',  # key in GeoJSON properties
    color_continuous_scale='Greens'
)
fig.update_geos(
    fitbounds="locations"
)
fig.update_layout(
    height=600,
    showlegend=True, title_text=f'Effect of {pollutant}'
)
fig.show()

# Heatmap of population

In [ ]:
# TODO: convert from state to national population data
fig = go.Figure()
fig = px.choropleth(
    pop_df, 
    geojson=mex_geojson, 
    color='POBTOT', 
    locations='NOM_ENT',  # column in state_pop_df that matches GeoJSON features
    featureidkey='properties.name',  # key in GeoJSON properties
    color_continuous_scale='Greens'
)
fig.update_geos(
    fitbounds="locations"
)
fig.update_layout(
    height=600,
    showlegend=True, title_text='Population heatmap'
)
fig.show()


# National mortality

In [22]:
# data source: https://www.inegi.org.mx/app/tabulados/interactivos/?pxq=mortalidad_Mortalidad_05_6a897e74-26b8-44de-8994-d9c46acdb64c
nat_mort = pd.read_excel('../Datos Nacionales/MORTALIDAD/Mortalidad_05.xlsx', skiprows=6)
nat_mort = nat_mort.set_index('Entidad federativa', drop=False)

# Grupo quinquenal de edad is all age groups
nat_mort_state_totals = nat_mort[nat_mort['Grupo quinquenal de edad'] == 'Total'].drop('Total')
print(nat_mort_state_totals.columns)


Index(['Entidad federativa', 'Grupo quinquenal de edad', '2010', '2011',
       '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', '2024'],
      dtype='object')


In [ ]:
fig = go.Figure()
fig = px.choropleth(
    nat_mort_state_totals, 
    geojson=mex_geojson, 
    color='2024', 
    locations='Entidad federativa',  # column in state_pop_df that matches GeoJSON features
    featureidkey='properties.name',  # key in GeoJSON properties
    color_continuous_scale='Greens'
)
fig.update_geos(
    fitbounds="locations"
)
fig.update_layout(
    height=600,
    showlegend=True, title_text='Mortality heatmap'
)
# fig.show()